# Step 2: Main notebook

In [12]:
import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point
from matplotlib import pyplot as plt

import os

from method_a_buffer import extract_buffer_feature
from method_b_catchment import extract_catchment_feature
from method_c_zonal import compute_zonal_stat
#from method_d_join import spatial_join_attribute

import geopandas as gpd
import folium
from folium import Choropleth, CircleMarker, GeoJson
import osmnx as ox
import branca.colormap as cm

## Imports

#### Import des segments

In [13]:
# Load segments GeoDataFrame (with 'segment_id')
print("Loading pedestrian segments, replace file path -->")
file_path = '../../Data/output/step-1'
segments_gdf = gpd.read_file(f"{file_path}/step1_pedestrian_segments.gpkg")

Loading pedestrian segments, replace file path -->


#### Import du csv - méthode de traitement des attributs

In [14]:
# #Attributes info
file_path = '../../Data/input'
# If the file attribuutes_info exists, load it
if os.path.exists(f"{file_path}/attributs_info.csv"):
    attributs_info = pd.read_csv(f"{file_path}/attributs_info.csv")
else:
    # If it does not exist, create a new DataFrame with the required structure
    attributs_info = pd.DataFrame(columns=['attribute', 'method', 'how', 'value_column', 'buffer_size'])
    
    # Example data to fill the DataFrame
    # You can modify this part to include the actual attributes you want to process
    attributs_info = pd.DataFrame({
        'attribute': ['arbre', 'accident'],
        'method': ['buffer', 'buffer'],
        'how' : ['count', 'sum'],  # 'how' can be 'count', 'mean', etc.
        'value_column': [None, 'PIETONS'],  # Column to aggregate
        'buffer_size': [30, 50]  # Example buffer size for method A
    })
    attributs_info.to_csv(f"{file_path}/attributs_info.csv", index=False)

#### Import des attributs et sauvegarde en gpkg (à modifier pour chaque attribut)

In [15]:
if not os.path.exists('../../Data/output/step-2/gpkg_attributs'):
    os.makedirs('../../Data/output/step-2/gpkg_attributs')
save_path = '../../Data/output/step-2/gpkg_attributs'


#Chargement de la couhe des accidents 
file_path = '../../Data/input/accident/OTC_ACCIDENTS-SHP'
accident_gdf  = gpd.read_file(f"{file_path}/OTC_ACCIDENTS.shp")
print('Couche accident chargée avec succès')
accident_gdf = accident_gdf.to_crs(2056)
accident_gdf.to_file(f"{save_path}/accident.gpkg", driver='GPKG')

#Chargement de la couche des arbre
file_path = '../../Data/input/arbre/SIPV_ICA_ARBRE_ISOLE-SHP'
arbre_gdf = gpd.read_file(f"{file_path}/SIPV_ICA_ARBRE_ISOLE.shp")
print('Couche arbre chargée avec succès')
arbre_gdf = arbre_gdf.to_crs(2056)
save_path = '../../Data/output/step-2/gpkg_attributs'
arbre_gdf.to_file(f"{save_path}/arbre.gpkg", driver='GPKG')

# Ajouter les autres couches d'attributs ici

Couche accident chargée avec succès
Couche arbre chargée avec succès


## Traitement des attributs

In [16]:
# # Charger la table des attributs et méthodes
# attributs_info = pd.read_csv('../../Data/input/attributs_info.csv')

# # Charger les segments
# segments_gdf = gpd.read_file('../../Data/output/step-1/step1_pedestrian_segments.gpkg')

# Garder uniquement les colonnes 'segment_id', 'vitesse', et 'geometry' dans segments_gdf
segments_gdf = segments_gdf[['osmid', 'maxspeed', 'geometry', 'segment_id']]

# Boucle sur chaque attribut
for _, row in attributs_info.iterrows():
    attribute_name = row['attribute']
    method = row['method']
    how = row['how']
    value_column = row['value_column']
    buffer_size = row['buffer_size']

    # Charger la couche attribut depuis gpd_attributs
    attribute_gdf = gpd.read_file(f"../../Data/output/step-2/gpkg_attributs/{attribute_name}.gpkg")
    attribute_gdf = attribute_gdf.to_crs(segments_gdf.crs)

    # Appliquer la méthode
    if method == "buffer":
        attribute_df = extract_buffer_feature(
            segments_gdf,
            attribute_gdf,
            feature_name=attribute_name,
            buffer_radius=buffer_size,
            how=how,
            value_column=value_column
        )
        print(f"Buffer feature extracted for {attribute_name} with method {method}")
    elif method == "catchment":
        attribute_df = extract_catchment_feature(
            segments_gdf=segments_gdf,
            feature_name=attribute_name,
            buffer_radius=buffer_size,
            points_gdf=attribute_gdf
        )
        print(f"Catchment feature extracted for {attribute_name} with method {method}")
    elif method == "zonal":
        attribute_df = compute_zonal_stat(
            segments_gdf,
            attribute_gdf,
            feature_name=attribute_name,
            value_column=value_column
        )
        print(f"Zonal statistics computed for {attribute_name} with method {method}")
    # Ajoute d'autres méthodes si besoin

    # Ajouter la colonne au GeoDataFrame principal
    segments_gdf[f'{attribute_name}_{method}_{buffer_size}'] = attribute_df[attribute_name]

# Sauvegarder le GeoDataFrame mis à jour
segments_gdf.to_csv('../../Data/output/step-2/step2_features.csv', index=False)

Buffer feature extracted for arbre with method buffer
Buffer feature extracted for accident with method buffer


In [17]:
pd.read_csv('../../Data/output/step-2/step2_features.csv').head(5)  # Afficher les 5 premières lignes pour vérification

,osmid,maxspeed,geometry,segment_id,arbre_buffer_30,accident_buffer_50
0,5196945,50,LINESTRING (2500419.1717844154 1120016.3567950...,0,16.0,3.0
1,29349156,50,LINESTRING (2500377.7763819178 1119989.4884629...,1,25.0,0.0
2,"[27938571, 1353905054, 1353905055]",50,LINESTRING (2500377.7763819178 1119989.4884629...,2,36.0,3.0
3,82882749,50,LINESTRING (2499812.609778903 1118296.38784988...,3,12.0,7.0
4,185888228,50,LINESTRING (2499812.609778903 1118296.38784988...,4,11.0,7.0
